# PTB-XL Visual Analytics en Colab

Ejecuta las celdas en orden. La plataforma se levanta dentro de Colab y se muestra aquí mismo. Trabaja con el conjunto de prueba (1.650 ECG).

## 1. Clonar el repositorio e instalar

In [ ]:
import os
if not os.path.exists('ptbxl-visual-analytics'):
    !git clone https://github.com/IEstremadoyro/ptbxl-visual-analytics.git
%cd ptbxl-visual-analytics
!pip -q install -r requirements.txt

## 2. Descargar los datos grandes (390 MB)

Baja `data/senales_norm_f16.npy` desde la sección *Releases* del repositorio.

In [ ]:
!python scripts/descargar_datos.py

## 3. Levantar la plataforma

In [ ]:
import subprocess, time, urllib.request
servidor = subprocess.Popen(['python', '-m', 'uvicorn', 'backend.main:app', '--port', '8000'],
                            stdout=open('servidor.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8000/api/coocurrencia', timeout=2); break
    except Exception:
        time.sleep(1)
print('Servidor listo' if servidor.poll() is None else open('servidor.log').read())

## 4. Ver la plataforma

Si el cuadro sale en blanco, ejecuta la celda 4b para abrirla en otra pestaña.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=950)

In [ ]:
# 4b. Alternativa: enlace para abrirla en otra pestaña
from google.colab import output
output.serve_kernel_port_as_window(8000)

## 5. (Opcional) Regenerar resultados

Vuelve a calcular vecinos, K-means/DBSCAN y el patrón de los infartos perdidos. Necesita torch y scikit-learn (ya vienen en Colab) y puede tardar unos minutos. Al terminar, reinicia la plataforma (celda 3) para ver los datos nuevos.

In [ ]:
!python scripts/generar_interpretabilidad.py
!python scripts/analisis_patron.py